# Content Refresh Prioritization — Capstone Research Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vikasbit/flyrank-content-refresh-prioritizatio/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This capstone notebook summarizes the research question, data, methodology, results, limitations, and action playbook for Content Refresh Prioritization.


## 1. Question

**Research Question:** Which webpages should be prioritized for content refresh based on historical search performance and content signals?

**Decision Supported:** Content teams have many webpages competing for limited review time. This project explores how historical search-performance signals and content signals can help prioritize pages that may warrant content-refresh review.


In [ ]:
import json
import pandas as pd

with open("outputs/summary.json") as f:
    summary = json.load(f)

print("Rows scored:", summary["rows_scored"])
print("Best model:", summary["best_model"])
print("Target positive rate:", round(summary["target_positive_rate"], 3))


## 2. Data

- Dataset: Bundled anonymized FlyRank dataset (`content_refresh_anonymized.csv`).
- Rows: 30,000 scored rows (27,675 train / 2,325 test split across 32 clients).
- Target: `is_declining_label` (54.21% base rate).
- Exclusions: Pseudonymized IDs (`content_id`, `client_id`), no private URLs or query text.


In [ ]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")


## 3. Methodology

- Feature Construction: 52 features (18 numeric, 8 categorical One-Hot encoded).
- Split Strategy: Client-holdout split (`client_holdout`).
- Models Evaluated: Baseline Rules, Logistic Regression, Decision Tree, Random Forest.
- Leakage Audit: Zero forward-window target leaks.


In [ ]:
with open("outputs/model_results.json") as f:
    res = json.load(f)

print("Split strategy:", res["split_strategy"])
print("Feature count:", res["feature_count"])


## 4. Results (vs baseline)

### Model Comparison Table

| Method | Validation | ROC AUC | Avg Precision | Precision@50 | Recall | F1 Score |
|---|---|---:|---:|---:|---:|---:|
| Week-4 Baseline | Client-holdout split | 0.627 | 0.468 | 0.240 | 0.189 | 0.274 |
| Decision Tree | Client-holdout split | 0.742 | 0.575 | 0.620 | 0.716 | 0.634 |
| Logistic Regression | Client-holdout split | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
| **Random Forest (Best)** | Client-holdout split | **0.747** | **0.610** | **0.680** | **0.741** | **0.638** |
| Week-6 Honest Validation | Grouped/time-aware | 0.747 | 0.610 | 0.680 | 0.741 | 0.638 |


In [ ]:
models = res["models"]
b = res["baseline"]
print(f"Baseline Precision@50: {b['baseline_precision_at_50']:.3f}")
print(f"Random Forest Precision@50: {models['random_forest']['precision_at_50']:.3f}")
print(f"Random Forest ROC AUC: {models['random_forest']['roc_auc']:.3f}")


## 5. Limitations

- Observational data only; no causal guarantees.
- Seasonal traffic variations not fully modeled in 90-day window.
- **Mandatory Statement:** *This analysis provides directional decision-support and does not establish that refreshing a page will cause improved traffic, rankings, or conversions.*


## 6. Ranked recommendations

- `CTR_OPPORTUNITY` → `REFRESH_CONTENT` (or `refresh_and_review_ctr`)
- `HIGH_DEMAND` → `REVIEW_HIGH_DEMAND` (or `refresh_and_review_engagement`)
- `RANKING_OPPORTUNITY` → `REVIEW_RANKING` (or `refresh`)
- `LOWER_PRIORITY` → `MONITOR` (or `monitor`)


In [ ]:
queue = pd.read_csv("outputs/refresh_queue.csv")
print("Suggested Actions Summary:")
print(queue["suggested_action"].value_counts())


## 7. Artifacts the paper embeds

- `outputs/charts/action_mix.svg`
- `outputs/charts/confidence_mix.svg`
- `outputs/charts/top_reason_codes.svg`
- `outputs/charts/top_feature_importance.svg`
- `outputs/charts/trend_distribution.svg`


# 5-Minute Demo Outline

## 1. Question — 45 seconds
Which webpages should be prioritized for content refresh based on historical search performance and content signals? Content teams have many webpages competing for limited review time. This project explores how historical search-performance signals and content signals can help prioritize pages that may warrant content-refresh review.

## 2. Method — 1 minute
- **Week-4 Baseline:** Hand-written rule flagging stale pages (updated >= 180 days ago) with high visibility (>= 500 impressions).
- **Week-5 Model:** Random Forest Classifier trained on 52 pre-decision features (GSC search visibility, GA4 engagement metrics, and article metadata).
- **Week-6 Honest Validation:** `client_holdout` split strategy where 20% of client domains (2,325 rows) are completely held out to evaluate true out-of-sample generalization.
- **Leakage Checks:** Target-derived trend signals (`trend_pct` and `trend_direction`) strictly omitted from feature matrices.

## 3. One Chart — 1 minute
![Top Feature Importance](outputs/charts/top_feature_importance.svg)
*Chart Explanation:* The feature importance breakdown reveals that `days_with_impressions` (16.06%) and `log_impressions_90d` (12.85%) dominate prediction over raw word count (4.12%). Regularity of search exposure is a stronger signal of stability than article length.

## 4. Honest Result — 1 minute
Under `client_holdout` validation split:
- **Baseline Rules:** Precision@50 = 0.240 | ROC AUC = 0.627
- **Random Forest Model:** Precision@50 = 0.680 | ROC AUC = 0.747 | Avg Precision = 0.610
*Honest Explanation:* The Random Forest model achieved a 2.8x precision improvement over baseline rules at Precision@50 (0.680 vs 0.240). These observed results provide a directional decision-support tool for editorial review, but do not establish that refreshing a page will cause improved traffic or rankings.

## 5. Recommendation — 1 minute
The action playbook maps probability scores and reason codes to reviewer categories:
- `CTR_OPPORTUNITY` → `REFRESH_CONTENT` (or `refresh_and_review_ctr`): High visibility pages in top positions with low CTR.
- `HIGH_DEMAND` → `REVIEW_HIGH_DEMAND` (or `refresh_and_review_engagement`): High traffic pages with low engagement/scroll rates.
- `RANKING_OPPORTUNITY` → `REVIEW_RANKING` (or `refresh`): Stale pages with strong baseline demand experiencing drop-off.
- `LOWER_PRIORITY` → `MONITOR` (or `monitor`): Stable or low-demand pages.
*Human Governance:* All recommendations are strictly decision-support reviewer aids; human editors must inspect and approve pages before taking action. Systems MUST NOT automatically publish, rewrite, delete, or redirect content.

# Shareable Cuts

## Social Post
🚀 Excited to share my FlyRank ML Internship capstone on Content Refresh Prioritization!

Content teams have many webpages competing for limited review time. To solve this capital allocation challenge, I built an ML decision-support pipeline that scores and ranks decaying pages for editorial refresh review.

Using 30,000 anonymized search performance rows across 32 clients, we evaluated 52 pre-decision signals (GSC search visibility, GA4 engagement metrics, and content metadata). Under strict client-holdout validation, our Random Forest classifier achieved a 0.747 ROC AUC and 0.680 Precision@50 — a 2.8x improvement over hand-written baseline rules (0.240 Precision@50).

The pipeline outputs a transparent reviewer queue mapping pages to actionable categories (CTR review, engagement review, refresh, monitor) while maintaining strict human-in-the-loop governance.

Check out the live research paper: https://vikasbit.github.io/flyrank-content-refresh-prioritizatio/
Built on the FlyRank ML Internship dataset (https://flyrank.ai).

## Employer-Facing Summary
I built a machine learning content refresh prioritization pipeline and research paper to address content decay and editorial audit inefficiency across client websites. Using 30,000 anonymized search performance rows from Google Search Console and GA4, I compared hand-written heuristic rules against Logistic Regression, Decision Tree, and Random Forest models under client-holdout validation. The analysis showed a 2.8x precision lift over baseline rules (0.680 vs 0.240 Precision@50, 0.747 ROC AUC), which I turned into a transparent reviewer queue and public research paper.
